# TinkerRL Offline Defense Demo

## Goal

Reproduce the defense-critical arithmetic from frozen local evidence without W&B, Hugging Face, a model endpoint, or any network connection. The notebook checks Claim 2 (matched-budget G=2 vs G=16), the corrected P4 completion-length contraction, the 983-vs-70+ run-count reconciliation, and the didactic ZVF fixture.

## Setup

Everything used below is package-relative. `run_checks.py` uses only Python's standard library, including direct XLSX/XML parsing. `data/manifest.json` pins SHA-256 for every copied evidence input.

In [1]:
from pathlib import Path
import sys

package_dir = Path.cwd()
if not (package_dir / 'run_checks.py').is_file():
    package_dir = (Path.cwd() / 'submission/demo/defense_fallback').resolve()
assert (package_dir / 'run_checks.py').is_file(), package_dir
sys.path.insert(0, str(package_dir))

from run_checks import (
    check_claim2, check_manifest, check_p4, check_run_audit,
    check_synthetic_fixture, run_all
)
print('Package:', package_dir)
print('Network required: no')

Package: /Users/arvind/Developer/agentic_repos/tinker-rl-lab/submission/demo/defense_fallback
Network required: no


## Steps

### 1. Verify copied evidence bytes and the synthetic ZVF fixture

The manifest check fails closed if any JSON, workbook, or figure changes. The synthetic fixture is deliberately didactic: it validates the ZVF calculation, not model quality.

In [2]:
manifest = check_manifest()
fixture = check_synthetic_fixture()
print(f"Manifest: {manifest['status']} ({manifest['file_count']} files)")
print(f"Fixture: ZVF={fixture['zvf']:.1f}, gradient utilization={fixture['gradient_utilization']:.1f}")

Manifest: PASS (14 files)
Fixture: ZVF=0.5, gradient utilization=0.5


### 2. Recompute Claim 2: matched-budget G=2 versus G=16

Each arm consumes 2,560 sampled completions: G=2 × batch 8 × 160 steps, or G=16 × batch 8 × 20 steps. We recompute last-10 reward and ZVF from the raw step logs.

In [3]:
claim2 = check_claim2()
print('run                         G  seed  steps  budget  late reward  late ZVF')
for row in claim2['rows']:
    print(f"{row['run']:<27} {row['group_size']:>2} {row['seed']:>5} {row['steps']:>6} {row['rollout_budget']:>7,} {row['late_reward_mean']:>12.6f} {row['late_zvf_mean']:>9.3f}")
print('\nDefense interpretation:', claim2['interpretation'])

run                         G  seed  steps  budget  late reward  late ZVF
er2b_g2_s123                 2   123    160   2,560     0.900000     0.975
er2b_g2_s456                 2   456    160   2,560     0.962500     0.975
er2b_g16_s123               16   123     20   2,560     0.321875     0.150
er2b_g16_s456               16   456     20   2,560     0.389062     0.100

Defense interpretation: At equal 2,560-rollout budgets, G=2 receives 160 optimizer steps and reaches the all-correct high-ZVF wall; G=16 receives 20 steps and remains mid-learning. This is a two-seed trajectory observation, not a universal group-size optimum.


### 3. Recompute P4's exact completion-length contraction

For each corrected 30-step arm, contraction is `(first-5 mean − last-10 mean) / first-5 mean`. This is the definition behind the reported approximately 3.8–12.2% range.

In [4]:
p4 = check_p4()
print('run                               loss     seed  first-5  last-10  contraction')
for row in p4['rows']:
    print(f"{row['run']:<34} {row['algorithm']:<8} {row['seed']:>4} {row['first5_mean_length']:>8.1f} {row['last10_mean_length']:>8.1f} {row['contraction_pct']:>10.4f}%")
print('\nDefense wording:', p4['defense_wording'])

run                               loss     seed  first-5  last-10  contraction
p4uncap_drgrpo_s123                Dr.GRPO   123    972.4    902.0     7.2333%
p4uncap_drgrpo_s42                 Dr.GRPO    42    999.4    931.4     6.8095%
p4uncap_drgrpo_s456                Dr.GRPO   456   1000.2    878.2    12.1950%
p4uncap_grpo_s123                  GRPO      123    980.5    943.6     3.7627%
p4uncap_grpo_s42                   GRPO       42   1004.0    905.0     9.8672%
p4uncap_grpo_s456                  GRPO      456    995.7    900.1     9.6045%

Defense wording: Completion length contracted by approximately 3.8-12.2% in all six corrected arms (first-5 mean to last-10 mean).


### 4. Reconcile 983 Tinker objects with the papers' 70+ corpus

The notebook reads `runs`, `key_runs`, and `insights` directly from the copied XLSX using only `zipfile` and XML parsing.

In [5]:
audit = check_run_audit()
print('Tinker training-client objects:', audit['tinker_run_objects'])
print('Curated paper corpus:', audit['curated_cross_library_claim'])
print('Claim-critical gold rows:', audit['claim_critical_gold_rows'])
print('\nReconciliation:', audit['plain_english'])

Tinker training-client objects: 983
Curated paper corpus: 70+ logged runs
Claim-critical gold rows: 19

Reconciliation: 983 is the broad infrastructure object count; 70+ is the curated cross-library telemetry corpus. They have different inclusion rules and are not competing totals.


## Checks

Run the complete fail-closed path once. Success also regenerates `output/report.json` and `output/dashboard.html`.

In [6]:
report = run_all()
assert report['status'] == 'PASS'
assert all(report[name]['status'] == 'PASS' for name in ('manifest', 'synthetic_fixture', 'claim2', 'p4', 'run_audit'))
print('ALL OFFLINE CHECKS: PASS')
print('Dashboard:', package_dir / 'output/dashboard.html')
print('Machine-readable report:', package_dir / 'output/report.json')

ALL OFFLINE CHECKS: PASS
Dashboard: /Users/arvind/Developer/agentic_repos/tinker-rl-lab/submission/demo/defense_fallback/output/dashboard.html
Machine-readable report: /Users/arvind/Developer/agentic_repos/tinker-rl-lab/submission/demo/defense_fallback/output/report.json


## Next Steps

1. Open `output/dashboard.html` and present the recomputed tables before the exported W&B figures.
2. Say **approximately 3.8–12.2%**, not 6–12%, for the P4 contraction.
3. Explain that 983 is the broad training-client object count, while 70+ is the curated cross-library corpus; their inclusion rules differ.
4. Keep the evidence limits explicit: two Claim 2 seeds per arm, three P4 seeds per loss, no new training here, and no causal or universal group-size claim.
5. If Jupyter is unavailable during the defense, run `./run.sh`; it is the stdlib-only authoritative fallback.